# 08 稳定性分析专题 (eda.stability + strategy_plots)

基于 `hscredit.xlsx` 真实信贷数据，演示完整的模型和特征稳定性监控体系。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import hscredit as hsc
from hscredit import (init_setting, WOEEncoder, LogisticRegression,
    feature_trend_by_time, feature_drift_comparison,
    population_drift_monitor, segment_scorecard_comparison,
    psi_analysis, batch_psi_analysis, csi_analysis,
)
from hscredit.core.eda.stability import (
    time_psi_tracking, stability_report, psi_cross_analysis,
)
from hscredit.core.metrics import psi, psi_table, psi_rating, batch_psi
from hscredit.core.metrics.stability import psi_rating
from hscredit.core.models.score_drift import ScoreDriftDetector
from hscredit.core.models.calibration import ModelCalibrator
from hscredit.core.metrics import ks, auc
init_setting()

df = pd.read_excel('hscredit.xlsx')
df['loan_date'] = pd.to_datetime(df['loan_date'], unit='D', origin='1899-12-30')
df['loan_month'] = df['loan_date'].dt.to_period('M').astype(str)
df['loan_quarter'] = df['loan_date'].dt.to_period('Q').astype(str)
target = 'FPD15'
features = ['bj_qy24','mobtech80','bairong_v1','lender_count_12m','overdue_loan_m1_count_12m',
    'loan_count_12m','loan_amt_sum_12m','network_loan_lender_count','last_performance_days']
X = df[features].fillna(-999); y = df[target]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
woe = WOEEncoder(max_n_bins=8)
X_woe_tr = woe.fit_transform(X_tr, y_tr)
X_woe_te = woe.transform(X_te)
lr = LogisticRegression(C=0.1, max_iter=1000); lr.fit(X_woe_tr, y_tr)
df['score'] = lr.predict_proba(woe.transform(X))[:,1]
print('准备完成')

## 1. 特征 PSI — 训练集 vs 测试集

In [ ]:
psi_result = batch_psi(X_tr, X_te, features=features)
psi_result['稳定性'] = psi_result['PSI'].apply(psi_rating)
print(psi_result)

## 2. 时序 PSI 追踪

In [ ]:
tracking = time_psi_tracking(df.fillna(-999), features[:5], 'loan_date', freq='Q')
if not tracking.empty:
    print(tracking.pivot_table(index='时间周期', columns='特征名', values='PSI值').round(4))

## 3. 综合稳定性报告

In [ ]:
rep = stability_report(df.fillna(-999), features[:6], 'loan_date', psi_threshold=0.1)
print(rep)

## 4. PSI 交叉矩阵（按季度）

In [ ]:
result = psi_cross_analysis(df.fillna(-999), ['bj_qy24','mobtech80'],
    date_col='loan_date', freq='Q', return_matrix=True)
print('bj_qy24 PSI矩阵:')
print(result['bj_qy24'].round(4))

## 5. 评分 PSI — 训练集 vs 测试集

In [ ]:
score_tr = lr.predict_proba(X_woe_tr)[:,1]
score_te = lr.predict_proba(X_woe_te)[:,1]
score_psi = psi(score_tr, score_te, max_n_bins=10)
print(f'评分PSI: {score_psi:.4f}  ({psi_rating(score_psi)})')
psi_table(score_tr, score_te, max_n_bins=10)

## 6. ScoreDriftDetector — 评分漂移监控

In [ ]:
detector = ScoreDriftDetector(n_bins=10, psi_threshold=0.1)
detector.fit(score_tr)
result = detector.detect(score_te)
print(f'PSI: {result["psi"]:.4f}, 状态: {result["drift_status"]}')
# 模拟强漂移
import numpy as np
shifted = np.clip(score_te + 0.15, 0, 1)
result2 = detector.detect(shifted)
print(f'漂移后PSI: {result2["psi"]:.4f}, 状态: {result2["drift_status"]}')

## 7. ModelCalibrator — 概率校准

In [ ]:
cal = ModelCalibrator(method='isotonic')
cal.fit(score_tr, y_tr)
score_cal = cal.transform(score_te)
print(f'校准前 KS:{ks(y_te,score_te):.4f}  AUC:{auc(y_te,score_te):.4f}')
print(f'校准后 KS:{ks(y_te,score_cal):.4f}  AUC:{auc(y_te,score_cal):.4f}')

## 8. feature_drift_comparison — 特征偏移图

In [ ]:
fig = feature_drift_comparison(X_tr, X_te, features, top_n=9)
plt.show()

## 9. population_drift_monitor — 多期偏移热力图

In [ ]:
quarters = sorted(df['loan_quarter'].unique())[:4]
df_list = [df[df['loan_quarter']==q].fillna(-999) for q in quarters if len(df[df['loan_quarter']==q])>50]
if len(df_list)>=2:
    fig = population_drift_monitor([d[features] for d in df_list], labels=[q for q,d in zip(quarters,df_list)],
        features=features[:6], top_n_drift=3)
    plt.show()